# ASC Input Format Ablation with RoBERTa


In [ ]:
!git clone https://github.com/TranTheHung2312332/FbOM-from-amazon-ds-PTIT.git

Cloning into 'FbOM-from-amazon-ds-PTIT'...
remote: Enumerating objects: 809, done.
remote: Counting objects: 100% (150/150), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 809 (delta 71), reused 73 (delta 27), pack-reused 659 (from 1)
Receiving objects: 100% (809/809), 63.45 MiB | 17.72 MiB/s, done.
Resolving deltas: 100% (327/327), done.
Updating files: 100% (194/194), done.


In [ ]:
import ast
import gc
import os
import re
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline,
    set_seed,
)
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

set_seed(42)

LABEL_ID_TO_NAME = {
    0: "negative",
    1: "neutral",
    2: "positive",
}

LABEL_NAME_TO_ID = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}

## 1. Configuration

In [ ]:
TRAIN_PATH = "/content/FbOM-from-amazon-ds-PTIT/data/gold_with_sentiments/gold_train.csv"
TEST_PATH = "/content/FbOM-from-amazon-ds-PTIT/data/gold_with_sentiments/gold_test.csv"

BASE_MODEL_NAME = "roberta-base"
ZERO_SHOT_MODEL_NAME = "roberta-large-mnli"

MAX_LENGTH = 192
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

ZERO_SHOT_BATCH_SIZE = 16
OUTPUT_DIR = "/content/asc_roberta_format_ablation"

SPECIAL_TOKENS = ["[ASP]", "[/ASP]"]

## 2. Load data

In [ ]:
def read_table(path):
    return pd.read_csv(path)

train_raw = read_table(TRAIN_PATH)
test_raw = read_table(TEST_PATH)

print("Train raw shape:", train_raw.shape)
print("Test raw shape:", test_raw.shape)

display(train_raw.head())

Train raw shape: (3200, 7)
Test raw shape: (800, 7)


,parent_asin,sentence_id,sentence_text,rating,aspects,sentiments,category_name
0,B01MA4YVNP,1,i took it to three different wall outlets and ...,1.0,"[""device""]",[0],electronics_p2
1,B0015BQ79Q,3,i put in dark knight on blu ray and any of my ...,5.0,"[""picture"", ""colors"", ""blacks""]","[2, 2, 2]",electronics_p2
2,B082VSBN2R,4,the sound is amazing and connected to my phone...,1.0,"[""sound"", ""connection to my phone""]","[2, 2]",electronics_p2
3,B00VANT7IW,6,if the author would have made her a little old...,5.0,"[""character""]",[0],kindle_store
4,B0BYRDPWT1,1,the chalk pen works well and washes off easily...,5.0,"[""chalk pen"", ""chalk pen""]","[""2"", ""2""]",office_products


## 3. Text preprocessing


In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def parse_list_cell(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    text = str(value).strip()

    if not text or text.lower() in {"nan", "none", "null"}:
        return []

    return ast.literal_eval(text)

In [ ]:
def preprocess_sentence_level(df):
    data = df.copy()

    data["sentence_text"] = data["sentence_text"].apply(clean_text)
    data["aspects"] = data["aspects"].apply(parse_list_cell)
    data["sentiments"] = data["sentiments"].apply(parse_list_cell)

    data = data[data["sentence_text"].str.len() > 0].reset_index(drop=True)
    data["has_aspect"] = data["aspects"].apply(lambda x: len(x) > 0)
    data = data[data["has_aspect"]].reset_index(drop=True)

    return data

train_sent = preprocess_sentence_level(train_raw)
test_sent = preprocess_sentence_level(test_raw)

print("Train rows with aspect:", train_sent.shape)
print("Test rows with aspect:", test_sent.shape)

display(train_sent.head())

Train rows with aspect: (1817, 8)
Test rows with aspect: (454, 8)


,parent_asin,sentence_id,sentence_text,rating,aspects,sentiments,category_name,has_aspect
0,B01MA4YVNP,1,i took it to three different wall outlets and ...,1.0,[device],[0],electronics_p2,True
1,B0015BQ79Q,3,i put in dark knight on blu ray and any of my ...,5.0,"[picture, colors, blacks]","[2, 2, 2]",electronics_p2,True
2,B082VSBN2R,4,the sound is amazing and connected to my phone...,1.0,"[sound, connection to my phone]","[2, 2]",electronics_p2,True
3,B00VANT7IW,6,if the author would have made her a little old...,5.0,[character],[0],kindle_store,True
4,B0BYRDPWT1,1,the chalk pen works well and washes off easily...,5.0,"[chalk pen, chalk pen]","[2, 2]",office_products,True


## 4. Build ASC instances


In [ ]:
def build_asc_instances(df):
    rows = []

    for _, row in df.iterrows():
        aspects = row["aspects"]
        sentiments = row["sentiments"]

        for aspect, sentiment in zip(aspects, sentiments):
            aspect = clean_text(aspect)

            if aspect:
                rows.append({
                    "parent_asin": row["parent_asin"],
                    "sentence_id": row["sentence_id"],
                    "sentence_text": row["sentence_text"],
                    "rating": row["rating"],
                    "aspect": aspect,
                    "label": int(sentiment),
                    "category_name": row["category_name"],
                })

    return pd.DataFrame(rows)

train_asc = build_asc_instances(train_sent)
test_asc = build_asc_instances(test_sent)

print("Train ASC shape:", train_asc.shape)
print("Test ASC shape:", test_asc.shape)

display(train_asc.head())

Train ASC shape: (3060, 7)
Test ASC shape: (782, 7)


,parent_asin,sentence_id,sentence_text,rating,aspect,label,category_name
0,B01MA4YVNP,1,i took it to three different wall outlets and ...,1.0,device,0,electronics_p2
1,B0015BQ79Q,3,i put in dark knight on blu ray and any of my ...,5.0,picture,2,electronics_p2
2,B0015BQ79Q,3,i put in dark knight on blu ray and any of my ...,5.0,colors,2,electronics_p2
3,B0015BQ79Q,3,i put in dark knight on blu ray and any of my ...,5.0,blacks,2,electronics_p2
4,B082VSBN2R,4,the sound is amazing and connected to my phone...,1.0,sound,2,electronics_p2


In [ ]:
print("Train label distribution")
display(train_asc["label"].map(LABEL_ID_TO_NAME).value_counts().to_frame("count"))

print("Test label distribution")
display(test_asc["label"].map(LABEL_ID_TO_NAME).value_counts().to_frame("count"))

Train label distribution


,count
label,
positive,1985
negative,932
neutral,143


Test label distribution


,count
label,
positive,496
negative,256
neutral,30


## 5. Input formats

Bốn input format được kiểm tra:

| Format | Input |
|---|---|
| `sentence_pair` | `sentence </s></s> aspect` |
| `aspect_marker` | `sentence` có `[ASP] aspect [/ASP]` |
| `marker_pair` | marked sentence + aspect |
| `auxiliary_sentence` | sentence + auxiliary sentence |


In [ ]:
def mark_aspect(sentence, aspect):
    pattern = re.compile(re.escape(aspect), flags=re.IGNORECASE)
    return pattern.sub(f"[ASP] {aspect} [/ASP]", sentence, count=1)

def apply_input_format(df, format_name):
    data = df.copy()

    if format_name == "sentence_pair":
        data["input_text"] = data["sentence_text"]
        data["input_pair"] = data["aspect"]
        data["zero_shot_text"] = data["sentence_text"] + " </s></s> " + data["aspect"]
        data["has_pair"] = True

    if format_name == "aspect_marker":
        data["input_text"] = [
            mark_aspect(sentence, aspect)
            for sentence, aspect in zip(data["sentence_text"], data["aspect"])
        ]
        data["input_pair"] = ""
        data["zero_shot_text"] = data["input_text"]
        data["has_pair"] = False

    if format_name == "marker_pair":
        data["input_text"] = [
            mark_aspect(sentence, aspect)
            for sentence, aspect in zip(data["sentence_text"], data["aspect"])
        ]
        data["input_pair"] = data["aspect"]
        data["zero_shot_text"] = data["input_text"] + " </s></s> " + data["aspect"]
        data["has_pair"] = True

    if format_name == "auxiliary_sentence":
        data["input_text"] = data["sentence_text"]
        data["input_pair"] = "The sentiment toward " + data["aspect"] + " is"
        data["zero_shot_text"] = data["sentence_text"] + " </s></s> " + data["input_pair"]
        data["has_pair"] = True

    return data

FORMAT_NAMES = [
    "sentence_pair",
    "aspect_marker",
    "marker_pair",
    "auxiliary_sentence",
]

In [ ]:
preview = apply_input_format(train_asc.head(5), "marker_pair")
display(preview[["sentence_text", "aspect", "input_text", "input_pair", "label"]])

,sentence_text,aspect,input_text,input_pair,label
0,i took it to three different wall outlets and ...,device,i took it to three different wall outlets and ...,device,0
1,i put in dark knight on blu ray and any of my ...,picture,i put in dark knight on blu ray and any of my ...,picture,2
2,i put in dark knight on blu ray and any of my ...,colors,i put in dark knight on blu ray and any of my ...,colors,2
3,i put in dark knight on blu ray and any of my ...,blacks,i put in dark knight on blu ray and any of my ...,blacks,2
4,the sound is amazing and connected to my phone...,sound,the [ASP] sound [/ASP] is amazing and connecte...,sound,2


## 6. Metrics

In [ ]:
def compute_basic_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)

    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )

    return {
        "accuracy": acc,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "weighted_precision": weighted_p,
        "weighted_recall": weighted_r,
        "weighted_f1": weighted_f1,
    }

In [ ]:
def make_report_df(y_true, y_pred):
    report = classification_report(
        y_true,
        y_pred,
        target_names=["negative", "neutral", "positive"],
        output_dict=True,
        zero_division=0,
    )
    return pd.DataFrame(report).T

def make_confusion_df(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    return pd.DataFrame(
        cm,
        index=["gold_negative", "gold_neutral", "gold_positive"],
        columns=["pred_negative", "pred_neutral", "pred_positive"],
    )

## 7. Zero-shot evaluation

In [ ]:
zero_shot_clf = pipeline(
    task="zero-shot-classification",
    model=ZERO_SHOT_MODEL_NAME,
    device=0 if torch.cuda.is_available() else -1,
)

candidate_labels = ["negative", "neutral", "positive"]
hypothesis_template = "The sentiment is {}."

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def predict_zero_shot(texts):
    predictions = []
    confidences = []

    for i in tqdm(range(0, len(texts), ZERO_SHOT_BATCH_SIZE)):
        batch = texts[i:i + ZERO_SHOT_BATCH_SIZE]

        outputs = zero_shot_clf(
            batch,
            candidate_labels=candidate_labels,
            hypothesis_template=hypothesis_template,
            multi_label=False,
            truncation=True,
        )

        if isinstance(outputs, dict):
            outputs = [outputs]

        for output in outputs:
            label_name = output["labels"][0]
            predictions.append(LABEL_NAME_TO_ID[label_name])
            confidences.append(float(output["scores"][0]))

    return np.array(predictions), np.array(confidences)

In [ ]:
zero_shot_results = []
zero_shot_predictions = {}

for format_name in FORMAT_NAMES:
    print("Zero-shot:", format_name)

    formatted_test = apply_input_format(test_asc, format_name)
    y_true = formatted_test["label"].to_numpy()

    y_pred, confidence = predict_zero_shot(formatted_test["zero_shot_text"].tolist())

    metrics = compute_basic_metrics(y_true, y_pred)
    metrics["setting"] = "zero_shot"
    metrics["format"] = format_name
    metrics["mean_confidence"] = float(confidence.mean())

    zero_shot_results.append(metrics)

    pred_df = formatted_test.copy()
    pred_df["prediction"] = y_pred
    pred_df["prediction_name"] = pred_df["prediction"].map(LABEL_ID_TO_NAME)
    pred_df["confidence"] = confidence
    zero_shot_predictions[format_name] = pred_df

zero_shot_summary = pd.DataFrame(zero_shot_results)
display(zero_shot_summary)

Zero-shot: sentence_pair


  0%|          | 0/49 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Zero-shot: aspect_marker


  0%|          | 0/49 [00:00<?, ?it/s]

Zero-shot: marker_pair


  0%|          | 0/49 [00:00<?, ?it/s]

Zero-shot: auxiliary_sentence


  0%|          | 0/49 [00:00<?, ?it/s]

,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,setting,format,mean_confidence
0,0.817136,0.620930,0.653175,0.626406,0.853291,0.817136,0.833237,zero_shot,sentence_pair,0.799393
1,0.796675,0.598475,0.617764,0.599554,0.833914,0.796675,0.813648,zero_shot,aspect_marker,0.792868
2,0.810742,0.612397,0.637486,0.615493,0.846793,0.810742,0.827049,zero_shot,marker_pair,0.787201
3,0.858056,0.651358,0.676843,0.661415,0.871839,0.858056,0.863892,zero_shot,auxiliary_sentence,0.821219


In [ ]:
for format_name, pred_df in zero_shot_predictions.items():
    print("\n" + "=" * 80)
    print("Zero-shot:", format_name)
    display(make_report_df(pred_df["label"], pred_df["prediction"]))
    display(make_confusion_df(pred_df["label"], pred_df["prediction"]))


Zero-shot: sentence_pair


,precision,recall,f1-score,support
negative,0.804688,0.804688,0.804688,256.000000
neutral,0.136364,0.300000,0.187500,30.000000
positive,0.921739,0.854839,0.887029,496.000000
accuracy,0.817136,0.817136,0.817136,0.817136
macro avg,0.620930,0.653175,0.626406,782.000000
weighted avg,0.853291,0.817136,0.833237,782.000000


,pred_negative,pred_neutral,pred_positive
gold_negative,206,25,25
gold_neutral,10,9,11
gold_positive,40,32,424



Zero-shot: aspect_marker


,precision,recall,f1-score,support
negative,0.790514,0.781250,0.785855,256.000000
neutral,0.104478,0.233333,0.144330,30.000000
positive,0.900433,0.838710,0.868476,496.000000
accuracy,0.796675,0.796675,0.796675,0.796675
macro avg,0.598475,0.617764,0.599554,782.000000
weighted avg,0.833914,0.796675,0.813648,782.000000


...

,precision,recall,f1-score,support
negative,0.802372,0.792969,0.797642,256.000000
neutral,0.121212,0.266667,0.166667,30.000000
positive,0.913607,0.852823,0.882169,496.000000
accuracy,0.810742,0.810742,0.810742,0.810742
macro avg,0.612397,0.637486,0.615493,782.000000
weighted avg,0.846793,0.810742,0.827049,782.000000


,pred_negative,pred_neutral,pred_positive
gold_negative,203,25,28
gold_neutral,10,8,12
gold_positive,40,33,423



Zero-shot: auxiliary_sentence


,precision,recall,f1-score,support
negative,0.827839,0.882812,0.854442,256.000000
neutral,0.190476,0.266667,0.222222,30.000000
positive,0.935760,0.881048,0.907580,496.000000
accuracy,0.858056,0.858056,0.858056,0.858056
macro avg,0.651358,0.676843,0.661415,782.000000
weighted avg,0.871839,0.858056,0.863892,782.000000


,pred_negative,pred_neutral,pred_positive
gold_negative,226,12,18
gold_neutral,10,8,12
gold_positive,37,22,437


## 8. Fine-tuning setup


```text
model = roberta-base
epochs = 3
learning_rate = 2e-5
max_length = 192
```

In [ ]:
def tokenize_dataset(df, tokenizer, has_pair):
    dataset = Dataset.from_pandas(
        df[["input_text", "input_pair", "label"]],
        preserve_index=False,
    )

    def tokenize_batch(batch):
        if has_pair:
            return tokenizer(
                batch["input_text"],
                batch["input_pair"],
                truncation=True,
                padding="max_length",
                max_length=MAX_LENGTH,
            )

        return tokenizer(
            batch["input_text"],
            truncation=True,
            padding="max_length",
            max_length=MAX_LENGTH,
        )

    tokenized = dataset.map(tokenize_batch, batched=True)
    tokenized = tokenized.rename_column("label", "labels")
    tokenized.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

    return tokenized

In [ ]:
def train_one_format(format_name):
    print("Fine-tune:", format_name)

    formatted_train = apply_input_format(train_asc, format_name)
    formatted_test = apply_input_format(test_asc, format_name)
    has_pair = bool(formatted_train["has_pair"].iloc[0])

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_NAME,
        num_labels=3,
        id2label=LABEL_ID_TO_NAME,
        label2id=LABEL_NAME_TO_ID,
    )

    if format_name in ["aspect_marker", "marker_pair"]:
        tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})
        model.resize_token_embeddings(len(tokenizer))

    train_ds = tokenize_dataset(formatted_train, tokenizer, has_pair)
    test_ds = tokenize_dataset(formatted_test, tokenizer, has_pair)

    args = TrainingArguments(
        output_dir=f"/content/asc_roberta_tmp/{format_name}",
        seed=42,
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        warmup_ratio=WARMUP_RATIO,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        fp16=torch.cuda.is_available(),
        logging_strategy="steps",
        logging_steps=50,
        save_strategy="no",
        eval_strategy="no",
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        processing_class=tokenizer,
    )

    trainer.train()

    TEMP_SAVE_DIR = f"/content/asc_roberta_saved_models/{format_name}"

    trainer.save_model(TEMP_SAVE_DIR)
    tokenizer.save_pretrained(TEMP_SAVE_DIR)

    print("Saved temporary model to:", TEMP_SAVE_DIR)

    pred_output = trainer.predict(test_ds)
    y_pred = pred_output.predictions.argmax(axis=-1)
    y_true = formatted_test["label"].to_numpy()

    metrics = compute_basic_metrics(y_true, y_pred)
    metrics["setting"] = "fine_tune"
    metrics["format"] = format_name

    pred_df = formatted_test.copy()
    pred_df["prediction"] = y_pred
    pred_df["prediction_name"] = pred_df["prediction"].map(LABEL_ID_TO_NAME)

    del model, tokenizer, trainer, train_ds, test_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return metrics, pred_df

## 9. Run fine-tuning experiments

In [ ]:
fine_tune_results = []
fine_tune_predictions = {}

for format_name in FORMAT_NAMES:
    metrics, pred_df = train_one_format(format_name)
    fine_tune_results.append(metrics)
    fine_tune_predictions[format_name] = pred_df

fine_tune_summary = pd.DataFrame(fine_tune_results)
display(fine_tune_summary)

Fine-tune: sentence_pair


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/3060 [00:00<?, ? examples/s]

Map:   0%|          | 0/782 [00:00<?, ? examples/s]

...

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
50,0.957617
100,0.639997
150,0.494178
200,0.444310
250,0.424953
300,0.422262
350,0.422792
400,0.386372
450,0.310988
500,0.334806


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved temporary model to: /content/asc_roberta_saved_models/auxiliary_sentence


,accuracy,macro_precision,macro_recall,macro_f1,weighted_precision,weighted_recall,weighted_f1,setting,format
0,0.865729,0.794946,0.609420,0.620095,0.858193,0.865729,0.852039,fine_tune,sentence_pair
1,0.874680,0.829915,0.624563,0.643990,0.869887,0.874680,0.862043,fine_tune,aspect_marker
2,0.884910,0.584743,0.604923,0.594615,0.850607,0.884910,0.867360,fine_tune,marker_pair
3,0.872123,0.581176,0.590012,0.584703,0.838605,0.872123,0.853939,fine_tune,auxiliary_sentence


In [ ]:
for format_name, pred_df in fine_tune_predictions.items():
    print("\n" + "=" * 80)
    print("Fine-tune:", format_name)
    display(make_report_df(pred_df["label"], pred_df["prediction"]))
    display(make_confusion_df(pred_df["label"], pred_df["prediction"]))


Fine-tune: sentence_pair


,precision,recall,f1-score,support
negative,0.837945,0.828125,0.833006,256.000000
neutral,0.666667,0.066667,0.121212,30.000000
positive,0.880228,0.933468,0.906067,496.000000
accuracy,0.865729,0.865729,0.865729,0.865729
macro avg,0.794946,0.609420,0.620095,782.000000
weighted avg,0.858193,0.865729,0.852039,782.000000


,pred_negative,pred_neutral,pred_positive
gold_negative,212,0,44
gold_neutral,9,2,19
gold_positive,32,1,463



Fine-tune: aspect_marker


,precision,recall,f1-score,support
negative,0.854839,0.828125,0.841270,256.00000
neutral,0.750000,0.100000,0.176471,30.00000
positive,0.884906,0.945565,0.914230,496.00000
accuracy,0.874680,0.874680,0.874680,0.87468
macro avg,0.829915,0.624563,0.643990,782.00000
weighted avg,0.869887,0.874680,0.862043,782.00000


...

,precision,recall,f1-score,support
negative,0.853846,0.867188,0.860465,256.00000
neutral,0.000000,0.000000,0.000000,30.00000
positive,0.900383,0.947581,0.923379,496.00000
accuracy,0.884910,0.884910,0.884910,0.88491
macro avg,0.584743,0.604923,0.594615,782.00000
weighted avg,0.850607,0.884910,0.867360,782.00000


,pred_negative,pred_neutral,pred_positive
gold_negative,222,0,34
gold_neutral,12,0,18
gold_positive,26,0,470



Fine-tune: auxiliary_sentence


,precision,recall,f1-score,support
negative,0.870833,0.816406,0.842742,256.000000
neutral,0.000000,0.000000,0.000000,30.000000
positive,0.872694,0.953629,0.911368,496.000000
accuracy,0.872123,0.872123,0.872123,0.872123
macro avg,0.581176,0.590012,0.584703,782.000000
weighted avg,0.838605,0.872123,0.853939,782.000000


,pred_negative,pred_neutral,pred_positive
gold_negative,209,0,47
gold_neutral,8,0,22
gold_positive,23,0,473


## 10. Final comparison

In [ ]:
all_results = pd.concat(
    [zero_shot_summary, fine_tune_summary],
    ignore_index=True,
)

metric_cols = [
    "setting",
    "format",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "macro_precision",
    "macro_recall",
    "weighted_precision",
    "weighted_recall",
]

display(
    all_results[metric_cols]
    .sort_values(["setting", "macro_f1"], ascending=[True, False])
    .reset_index(drop=True)
)

,setting,format,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,weighted_precision,weighted_recall
0,fine_tune,aspect_marker,0.874680,0.643990,0.862043,0.829915,0.624563,0.869887,0.874680
1,fine_tune,sentence_pair,0.865729,0.620095,0.852039,0.794946,0.609420,0.858193,0.865729
2,fine_tune,marker_pair,0.884910,0.594615,0.867360,0.584743,0.604923,0.850607,0.884910
3,fine_tune,auxiliary_sentence,0.872123,0.584703,0.853939,0.581176,0.590012,0.838605,0.872123
4,zero_shot,auxiliary_sentence,0.858056,0.661415,0.863892,0.651358,0.676843,0.871839,0.858056
5,zero_shot,sentence_pair,0.817136,0.626406,0.833237,0.620930,0.653175,0.853291,0.817136
6,zero_shot,marker_pair,0.810742,0.615493,0.827049,0.612397,0.637486,0.846793,0.810742
7,zero_shot,aspect_marker,0.796675,0.599554,0.813648,0.598475,0.617764,0.833914,0.796675


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!rm -rf /content/drive/MyDrive/absa_self_train_phase1/asc_teacher_phase1
!cp -r /content/asc_roberta_saved_models/aspect_marker /content/drive/MyDrive/absa_self_train_phase1/asc_teacher_phase1

print("Copied best model to Drive.")

Copied best model to Drive.
